# 03b — Consumer heterogeneity: income and business/leisure types

Two extensions of the random-coefficients model of notebook 03, on the same estimation sample of routes:

1. **Income.** Each market's consumers are drawn from the population-weighted mixture of the two endpoint
   metros' household income distributions (ACS 2020–2024). The lognormal price coefficient becomes
   $\alpha_i = \exp(\pi_0 + \pi_y y_i + \sigma_p \nu_i)$ with $y_i$ = log income relative to \$75k, and income also
   shifts the taste for nonstop service. With aggregate data alone, $\pi_y$ is identified from how substitution
   patterns vary with the income distribution across markets (the course notes' "demographics in BLP").
2. **Business and leisure travellers** (Berry, Carnall and Spiller 2006; Berry and Jia 2010). Two discrete types
   with population shares $\lambda$ and $1-\lambda$: type-specific price sensitivity, value of flying and value of
   nonstop service. Type weights are not estimated by PyBLP, so $\lambda$ is profiled over a grid and chosen by the
   GMM objective.

DB1B records no traveller characteristics and no second choices, so micro moments of the Conlon–Gortmaker
type cannot be formed from it; that is the main limit on identifying these parameters.

In [ ]:
%run 00_setup.ipynb

In [ ]:
%run 00_model_functions.ipynb

In [ ]:
IV = IV_SETS["baseline"]
p = pd.read_parquet(INTERIM / "products_airport_nondir_iv.parquet")
est = load_json("demand_full_sample")
rc_routes = est["rc_routes"][: (80 if SMOKE else None)]
p_rc = p[p.route.isin(rc_routes)].reset_index(drop=True)
brackets = pd.read_parquet(CLEAN / "acs_income_brackets.parquet")
edges = [0, 10, 15, 20, 25, 30, 35, 40, 45, 50, 60, 75, 100, 125, 150, 200, 400]
INCOME = (brackets, edges)
base = est["rc_demand"]
print(f"RC sample {p_rc.market_ids.nunique():,} markets; baseline RC sigma {np.round(base['sigma'], 3)}, pi0 {base['pi'][1][0]:.3f}")

## 1. Income random coefficients

In [ ]:
prob_i = make_problem(p_rc, "rc_income", IV, income=INCOME)
pi0 = np.array([[0.0, 0.0], [base["pi"][1][0], -0.3], [0.0, 0.2]])
with pyblp.parallel(N_CORES):
    rc_i = solve_rc(prob_i, "rc_income", sigma0=base["sigma"], pi0=pi0, method="2s")
print(rc_i)

In [ ]:
pi_y = float(rc_i.pi[1, 1]); pi_0 = float(rc_i.pi[1, 0])
inc_grid = pd.Series([25, 50, 75, 100, 150, 250], name="household income ($000)")
alpha_by_income = pd.DataFrame({"median price sensitivity (per $100)": np.exp(pi_0 + pi_y * np.log(inc_grid / 75)),
                                "relative to $75k": np.exp(pi_y * np.log(inc_grid / 75))}, index=inc_grid)
save_table(alpha_by_income, "alpha_by_income", float_format="%.3f",
           note=f"Median price coefficient $\\exp(\\pi_0+\\pi_y\\ln(y/75))$ implied by the income model; $\\pi_y$ = {pi_y:.3f} "
                f"(s.e. {float(rc_i.pi_se[1, 1]):.3f}).")
alpha_by_income

## 2. Business and leisure types (profiled business share)

In [ ]:
grid = [0.1, 0.2, 0.3, 0.4, 0.5][: (2 if SMOKE else None)]
type_runs = []
for lam in grid:
    prob_t = make_problem(p_rc, "rc_types", IV, business_share=lam)
    pi_start = np.array([[0.0, 0.5], [base["pi"][1][0], -1.0], [0.0, 0.5]])
    with pyblp.parallel(N_CORES):
        r = solve_rc(prob_t, "rc_types", pi0=pi_start, method="1s")
    type_runs.append({"business share": lam, "objective": objective(r), "pi": np.asarray(r.pi).tolist(), "res": r})
    print(lam, round(objective(r), 3), np.round(r.pi, 3).tolist())
best_t = min(type_runs, key=lambda d: d["objective"])
LAMBDA = best_t["business share"]
prob_t = make_problem(p_rc, "rc_types", IV, business_share=LAMBDA)
with pyblp.parallel(N_CORES):
    rc_t = solve_rc(prob_t, "rc_types", pi0=best_t["pi"], method="2s")
print(rc_t)
prof = pd.DataFrame([{k: v for k, v in d.items() if k in ("business share", "objective")} for d in type_runs]).set_index("business share")

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 3.0))
ax.plot(prof.index, prof.objective, marker="o")
ax.set_xlabel("Business-traveller share $\\lambda$"); ax.set_ylabel("GMM objective (one-step)")
fig.tight_layout(); savefig(fig, "business_share_profile")
P = np.asarray(rc_t.pi)
types = pd.DataFrame({
    "price sensitivity (per $100)": [np.exp(P[1, 0] + P[1, 1]), np.exp(P[1, 0])],
    "extra value of flying": [P[0, 1], 0.0],
    "extra value of nonstop": [P[2, 1], 0.0],
    "population share": [LAMBDA, 1 - LAMBDA]}, index=["Business", "Leisure"])
save_table(types, "types_model", float_format="%.3f",
           note="Two-type model: price coefficient $-\\exp(\\pi_0+\\pi_b\\,\\mathrm{business})$; business share chosen by profiling the GMM "
                "objective over a grid.")
types

## 3. Implications on the full sample

In [ ]:
models = {"RC (baseline)": ("rc", {"sigma": base["sigma"], "pi": base["pi"]}, {}),
          "RC + income": ("rc_income", {"sigma": np.diag(rc_i.sigma).tolist(), "pi": np.asarray(rc_i.pi).tolist()}, {"income": INCOME}),
          "Business/leisure types": ("rc_types", {"sigma": [0.0, 0.0, 0.0], "pi": P.tolist()}, {"business_share": LAMBDA})}
p_eval = p if not SMOKE else p[p.route.isin(rc_routes)].reset_index(drop=True)
post = {}
for name, (kind, prm, kw) in models.items():
    with pyblp.parallel(N_CORES):
        r = evaluate_fixed(p_eval, kind, prm, IV, **kw)
    post[name] = post_estimation(r, p_eval)
summ = pd.DataFrame({k: summarize_post(v) for k, v in post.items()})
save_table(summ, "heterogeneity_summary", float_format="%.3f",
           note="Full-sample elasticities, markups and diversion ratios under the three random-coefficient specifications.")
summ

In [ ]:
coef = coef_rows({
    "RC + income": [("pi_0 (log median price sensitivity)", float(rc_i.pi[1, 0]), float(rc_i.pi_se[1, 0])),
                    ("pi_y: log income x price", float(rc_i.pi[1, 1]), float(rc_i.pi_se[1, 1])),
                    ("pi: log income x nonstop", float(rc_i.pi[2, 1]), float(rc_i.pi_se[2, 1])),
                    ("sigma: constant", float(rc_i.sigma[0, 0]), float(rc_i.sigma_se[0, 0])),
                    ("sigma: log price coefficient", float(rc_i.sigma[1, 1]), float(rc_i.sigma_se[1, 1])),
                    ("sigma: nonstop", float(rc_i.sigma[2, 2]), float(rc_i.sigma_se[2, 2]))],
    "Business/leisure": [("pi_0 (log price sensitivity, leisure)", float(P[1, 0]), float(rc_t.pi_se[1, 0])),
                         ("business x log price sensitivity", float(P[1, 1]), float(rc_t.pi_se[1, 1])),
                         ("business x constant", float(P[0, 1]), float(rc_t.pi_se[0, 1])),
                         ("business x nonstop", float(P[2, 1]), float(rc_t.pi_se[2, 1]))]})
save_table(coef, "heterogeneity_estimates", note=f"PyBLP two-step GMM on the RC estimation sample; business share {LAMBDA}.")
coef

In [ ]:
if True:
    save_json({"rc_income": rc_params(rc_i), "rc_types": {**rc_params(rc_t), "business_share": LAMBDA},
               "profile": prof.reset_index().to_dict("records"), "summary": summ.to_dict()}, "demand_heterogeneity")